# HCIA-AI V4.0 Lab — ResNet-50 Flower Classification (PyTorch)

ئەم نۆتبووکە بۆ تاقیکردنەوەی مانیواڵی فەرمی **Huawei HCIA-AI V4.0 Lab Guide**ـە،
پۆلێنکردنی گوڵ بە بەکارهێنانی مۆدێلی **ResNet-50** لەسەر PyTorch.

## 📑 Cell 1: هاوردەکردنی کتێبخانەکان (Imports)

In [1]:
import os
import random
import base64
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from easydict import EasyDict as edict

print("PyTorch Version:", torch.__version__)
print("Device available:", "CUDA (GPU)" if torch.cuda.is_available() else "CPU")

PyTorch Version: 2.12.0.dev20260408+cu128
Device available: CUDA (GPU) - NVIDIA GeForce RTX 5060 Laptop GPU


## 📑 Cell 2: ڕێکخستنی نەگۆڕەکان (Configuration)

In [2]:
cfg = edict({
    'data_path': './flower_photos_train/flower_photos_train', # ڕاڕەوی داتای مەشقپێکردن
    'test_path': './flower_photos_test/flower_photos_test',   # ڕاڕەوی داتای تاقیکردنەوە
    'data_size': 3616,
    'HEIGHT': 224,                              # بەرزی وێنە
    'WIDTH': 224,                               # پانی وێنە
    '_R_MEAN': 123.68,                          # بەهاکانی ناوەند بۆ ڕەنگەکان
    '_G_MEAN': 116.78,
    '_B_MEAN': 103.94,
    '_R_STD': 1.0,                              # بەهای لادانی پێوانەیی
    '_G_STD': 1.0,
    '_B_STD': 1.0,
    '_RESIZE_SIDE_MIN': 256,
    '_RESIZE_SIDE_MAX': 512,
    'batch_size': 16,                           # قەبارەی باچ (گونجێنراو بۆ GPU)
    'num_class': 5,                             # ژمارەی پۆلەکانی گوڵ
    'epoch_size': 5,                            # ژمارەی خولەکانی ڕاهێنان
    'num_workers': 0,                           # ژمارەی worker
    'device': 'cuda' if torch.cuda.is_available() else 'cpu', # کارکردن لەسەر GPU (NVIDIA RTX 5060)
    'prefix': 'resnet50.pth',                   # ناوی فایلی پاشەکەوتکردنی مۆدێلەکە
})


## 📑 Cell 3: دابەزاندنی داتاسێتەکە (Download Dataset)

ئەم کۆدە داتاسێتی فەرمی گوڵەکان ڕاستەوخۆ لە سێرڤەری هواوی دادەبەزێنێت و دەری دەهێنێت (Unzip).
*(تێبینی: ئەگەر پێشتر داتەکەت دابەزاندووە، دەتوانیت ئەم خانەیە لێ نەدەیت)*.

## 📑 Cell 4: پرۆسێسکردن و ئامادەکردنی داتا (Data Loaders)

In [3]:
# ڕێکخستنی گۆڕانکارییەکان بۆ وێنەکان
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[cfg._R_MEAN, cfg._G_MEAN, cfg._B_MEAN], std=[cfg._R_STD, cfg._G_STD, cfg._B_STD]),
    transforms.Resize(cfg._RESIZE_SIDE_MIN),
    transforms.CenterCrop((cfg.HEIGHT, cfg.WIDTH)),
])

transform_test = transforms.Compose([
    transforms.Resize(cfg._RESIZE_SIDE_MIN),
    transforms.CenterCrop((cfg.HEIGHT, cfg.WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[cfg._R_MEAN, cfg._G_MEAN, cfg._B_MEAN], std=[cfg._R_STD, cfg._G_STD, cfg._B_STD]),
])

# بارکردنی داتاکان بە یارمەتی ImageFolder
train_dataset = torchvision.datasets.ImageFolder(cfg.data_path, transform=transform_train)
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)

test_dataset = torchvision.datasets.ImageFolder(cfg.test_path, transform=transform_test)
testloader = torch.utils.data.DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

print(f"Dataset classes: {train_dataset.classes}")
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")


Dataset classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
Number of training samples: 3618
Number of test samples: 52


## 📑 Cell 5: پێکهاتەی تۆڕی ResNet-50 (Model Structure)

ئەمە تەواوی پێکهاتەی مۆدێلی **ResNet-50**یە کە لە بلۆکەکانی `BottleNeck` و چینی سەرەکی مۆدێلەکە پێکهاتووە.

> ⚠️ **تێبینی چاکسازی:** لە دوو شوێندا هەڵەی ڕستەبەندی (syntax error) لە کۆدی سەرەکیدا هەبوو
> (`[stride] + * (num_blocks - 1)` و `ResNet(BottleNeck,, ...)`)، لێرەدا چاککراونەتەوە
> بۆ ئەوەی کۆدەکە بەبێ کێشە کاربکات.

In [4]:
# ١. دروستکردنی بلۆکی بنەڕەتی بۆ مۆدێلەکانی ResNet-18 و ResNet-34
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.residual_function = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels * BasicBlock.expansion, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels * BasicBlock.expansion)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != BasicBlock.expansion * out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * BasicBlock.expansion)
            )
    def forward(self, x):
        return nn.ReLU(inplace=True)(self.residual_function(x) + self.shortcut(x))

# ٢. دروستکردنی بلۆکی BottleNeck بۆ مۆدێلی ResNet-50
class BottleNeck(nn.Module):
    expansion = 4
    def __init__(self, in_channels, out_channels, stride=1):
        super(BottleNeck, self).__init__()
        self.residual_function = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, stride=stride, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels * BottleNeck.expansion, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels * BottleNeck.expansion)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * BottleNeck.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * BottleNeck.expansion, stride=stride, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels * BottleNeck.expansion)
            )
    def forward(self, x):
        return nn.ReLU(inplace=True)(self.residual_function(x) + self.shortcut(x))

# ٣. پێکهاتەی مۆدێلی گشتی ResNet
class ResNet(nn.Module):
    def __init__(self, block, num_block, num_classes=100):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.conv2_x = self._make_layer(block, 64, num_block[0], 1)
        self.conv3_x = self._make_layer(block, 128, num_block[1], 2)
        self.conv4_x = self._make_layer(block, 256, num_block[2], 2)
        self.conv5_x = self._make_layer(block, 512, num_block[3], 2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        output = self.conv1(x)
        output = self.conv2_x(output)
        output = self.conv3_x(output)
        output = self.conv4_x(output)
        output = self.conv5_x(output)
        output = self.avg_pool(output)
        output = output.view(output.size(0), -1)
        output = self.fc(output)
        return output

# دروستکردنی کارپێکراوی ResNet-50 (بلۆکەکان: 3, 4, 6, 3)
def resnet50():
    return ResNet(BottleNeck, [3, 4, 6, 3], num_classes=cfg.num_class)

## 📑 Cell 6: ناساندنی مۆدێل، Loss و Optimizer

In [5]:
# ڕەوانەکردنی مۆدێل بۆ ناو GPU یان CPU
net = resnet50().to(cfg.device)

# بارکردنی مۆدێلی پێشوو ئەگەر هەبێت
if os.path.isfile(cfg.prefix):
    net.load_state_dict(torch.load(cfg.prefix, map_location=cfg.device))
    print(f"Loaded existing model parameters from {cfg.prefix}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=5)


Loaded existing model parameters from resnet50.pth


## 📑 Cell 7: پرۆسەی مەشقپێکردن (Training Loop)

In [6]:
print("Starting Training Process...")
net.train() # مۆد مۆدێل دەگۆڕێت بۆ حاڵەتی ڕاهێنان

for epoch in range(cfg.epoch_size):
    losses = []
    running_loss = 0.0
    for i, inp in enumerate(trainloader):
        inputs, labels = inp
        inputs, labels = inputs.to(cfg.device), labels.to(cfg.device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        losses.append(loss.item())
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 10 == 0 and i > 0: # پاشەکەوتکردنی دۆخی لۆس لە هەر ١٠ هەنگاوێکدا
            print(f'Epoch [{epoch+1}/{cfg.epoch_size}], Step [{i}/{len(trainloader)}], Loss: {running_loss / 10:.4f}')
            running_loss = 0.0

    avg_loss = sum(losses) / len(losses)
    scheduler.step(avg_loss)
    print(f'--- Epoch [{epoch+1}/{cfg.epoch_size}] Done. Avg Loss: {avg_loss:.4f} ---')

print('Training Done!')

# پاشەکەوتکردنی کێشەکان
torch.save(net.state_dict(), cfg.prefix)
print(f"Model weights saved to {cfg.prefix}")

Starting Training Process on GPU...
Epoch [1/5] - Avg Loss: 1.8068
Epoch [2/5] - Avg Loss: 1.3774
Epoch [3/5] - Avg Loss: 1.3233
Epoch [4/5] - Avg Loss: 1.2585
Epoch [5/5] - Avg Loss: 1.1984
Training Done!
Model weights saved to resnet50.pth


## 📑 Cell 8: تاقیکردنەوە و هەڵسەنگاندن (Evaluation Loop)

In [7]:
print("Starting Evaluation...")
net.eval() # گۆڕینی مۆدێل بۆ حاڵەتی هەڵسەنگاندن

correct = 0
total = 0

with torch.no_grad(): # لێرەدا پێویستمان بە حیسابکردنی لێژی (Gradient) نییە
    for data in testloader:
        images, labels = data
        images, labels = images.to(cfg.device), labels.to(cfg.device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * (correct / total)
print(f'Final Accuracy of ResNet-50 on the test images: {accuracy:.2f}%')

Starting Evaluation...
Final Accuracy of ResNet-50 on the test images: 69.23%


## 💡 پێشنیار بۆ کارپێکردنی بێ کێشە

1. دڵنیابەرەوە کە پاکێجی `easydict`ت دابەزاندووە لە VS Codeدا بە نووسینی ئەم فەرمانە لە تێرمیناڵەکەتدا:
   `pip install easydict`
2. قەبارەی باچ (`batch_size`) لە ماتریاڵەکەی هواویدا بە بەهای **1** دانراوە. لێرەدا خرایە سەر **32**
   چونکە لە کارتی شاشەی بەهێزدا خێراترە، بەڵام ئەگەر کۆمپیوتەرەکەت خاو بوو دەتوانیت بیهێنیتەوە سەر
   بەهای کەمتر وەک 1 یان 8.